# SWELL-KW Cognitive Load Analysis — Colab Edition

Self-contained. No dependency on any other dataset or local module —
everything needed is either a standard library (pandas/numpy/sklearn/
matplotlib/scipy) or defined in this notebook.

**What SWELL-KW is:** 25 people did real knowledge work (writing, email,
research) under 3 deliberately different conditions — Neutral,
Interruptions (forced emails), Time pressure. Behavior (mouse/keyboard
counts) and physiology (heart rate, heart rate variability, skin
conductance) were recorded per minute throughout.

**What this notebook tests:** does behavior alone carry real information
about cognitive load, and do behavior and physiology share a common
underlying signal even though they're measured completely differently?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
candidates = glob.glob('/content/drive/**/Behavioral-features - per minute.xlsx', recursive=True)
print("Found candidate files:")
for c in candidates:
    print(" ", c)

# If the search above found nothing, or found the wrong file, set this
# manually to the exact path on your Drive:
DATA_FILE = candidates[0] if candidates else "/content/drive/MyDrive/EDIT_THIS_PATH/Behavioral-features - per minute.xlsx"
print(f"\nUsing: {DATA_FILE}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.cross_decomposition import CCA
from sklearn.model_selection import ShuffleSplit
import warnings
warnings.filterwarnings("ignore")

HCI_COLS = [
    "SnMouseAct", "SnLeftClicked", "SnRightClicked", "SnDoubleClicked",
    "SnWheel", "SnDragged", "SnMouseDistance", "SnKeyStrokes", "SnChars",
    "SnSpecialKeys", "SnDirectionKeys", "SnErrorKeys", "SnShortcutKeys",
    "SnSpaces", "SnAppChange", "SnTabfocusChange",
    "CharactersRatio", "ErrorKeyRatio",
]
PHY_COLS = ["HR", "RMSSD", "SCL"]
COND_ORDER = ["N", "I", "T"]
COND_NAMES = {"N": "Neutral", "I": "Interruptions", "T": "Time pressure"}

In [ ]:
df = pd.read_excel(DATA_FILE)
df = df[~df["Condition"].isin({"R"})].copy()
for c in PHY_COLS:
    df[c] = df[c].replace(999, np.nan)

print(f"Participants: {df['PP'].nunique()}  |  Rows: {len(df)}")
print(df["Condition"].value_counts())

## Experiment A — Condition-level HCI <-> Physiology correlation

Aggregated per (participant x condition), per-user z-scored. Tests
whether specific behaviors and specific physiological measures move
together across the three conditions.

In [ ]:
agg = df.groupby(["PP", "Condition"])[HCI_COLS + PHY_COLS].mean().reset_index()
agg = agg[agg["Condition"].isin(COND_ORDER)]
for col in HCI_COLS + PHY_COLS:
    agg[col + "_z"] = agg.groupby("PP")[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-9))

print(f"Condition-level rows: {len(agg)}  (25 participants x 3 conditions = 75)\n")

results_a = []
for hci in HCI_COLS:
    for phy in PHY_COLS:
        x = agg[hci + "_z"].to_numpy(float)
        y = agg[phy + "_z"].to_numpy(float)
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 20:
            continue
        results_a.append((hci, phy, np.corrcoef(x[m], y[m])[0, 1]))

df_r = pd.DataFrame(results_a, columns=["HCI", "Phy", "r"])
pivot = df_r.pivot(index="HCI", columns="Phy", values="r")
print("Pearson r (condition-level):")
print(pivot.round(3))

strong = df_r[df_r["r"].abs() > 0.3].sort_values("r", key=abs, ascending=False)
print("\nStrongest pairs (|r| > 0.3):")
print(strong.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 9))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdBu_r", vmin=-0.6, vmax=0.6)
ax.set_xticks(range(len(PHY_COLS))); ax.set_xticklabels(PHY_COLS)
ax.set_yticks(range(len(HCI_COLS))); ax.set_yticklabels(pivot.index, fontsize=8)
for i in range(len(HCI_COLS)):
    for j in range(len(PHY_COLS)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(v) > 0.4 else "black")
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("HCI <-> Physiology at condition level")
plt.tight_layout(); plt.show()

## Experiment B — HCI -> Condition classifier (LOSO)

Can behavior alone tell you which condition (Neutral/Interruptions/
Time pressure) someone was in? Compared directly against the same
classifier trained on physiology instead, same folds, same evaluation.

In [ ]:
def loso_classify(X, y, groups, model_fn):
    subs = sorted(set(groups))
    accs, f1s = [], []
    for held in subs:
        tr = groups != held; te = groups == held
        if tr.sum() < 10 or te.sum() < 1:
            continue
        Xtr, ytr = X[tr], y[tr]; Xte, yte = X[te], y[te]
        mu = np.nanmean(Xtr, axis=0); sd = np.nanstd(Xtr, axis=0) + 1e-9
        Xtr = np.nan_to_num((Xtr - mu) / sd); Xte = np.nan_to_num((Xte - mu) / sd)
        m = model_fn(); m.fit(Xtr, ytr)
        pred = m.predict(Xte)
        accs.append(accuracy_score(yte, pred))
        f1s.append(f1_score(yte, pred, average="macro", zero_division=0))
    return np.array(accs), np.array(f1s)

groups = df["PP"].to_numpy()
y_all = df["Condition"].map({"N": 0, "I": 1, "T": 2}).to_numpy(dtype=int)
X_hci = np.nan_to_num(df[HCI_COLS].to_numpy(dtype=float))
X_phy = np.nan_to_num(df[PHY_COLS].to_numpy(dtype=float))

rf = lambda: RandomForestClassifier(200, min_samples_leaf=3,
                                     class_weight="balanced", random_state=0, n_jobs=-1)

hci_accs, hci_f1s = loso_classify(X_hci, y_all, groups, rf)
phy_accs, phy_f1s = loso_classify(X_phy, y_all, groups, rf)

print(f"HCI -> RF   acc={hci_accs.mean():.3f}  macro-F1={hci_f1s.mean():.3f}")
print(f"PHY -> RF   acc={phy_accs.mean():.3f}  macro-F1={phy_f1s.mean():.3f}")
print(f"Chance      acc=0.333")

all_true, all_pred = [], []
for held in sorted(set(groups)):
    tr = groups != held; te = groups == held
    Xtr, ytr = X_hci[tr], y_all[tr]; Xte, yte = X_hci[te], y_all[te]
    mu = Xtr.mean(0); sd = Xtr.std(0) + 1e-9
    m = rf(); m.fit((Xtr - mu) / sd, ytr)
    pred = m.predict((Xte - mu) / sd)
    all_true.extend(yte.tolist()); all_pred.extend(pred.tolist())
cm = confusion_matrix(all_true, all_pred)
print("\nConfusion matrix (rows=true, cols=pred), order N/I/T:")
print(cm)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(hci_accs))
ax.bar(x - 0.2, hci_accs, 0.4, label="HCI", color="steelblue")
ax.bar(x + 0.2, phy_accs[:len(hci_accs)], 0.4, label="PHY", color="tomato")
ax.axhline(1/3, color="k", ls="--", label="chance")
ax.set_xlabel("fold (participant)"); ax.set_ylabel("accuracy"); ax.legend()
ax.set_title("Condition classification, per fold")
plt.tight_layout(); plt.show()

## Experiment C — Direction prediction (is HR/RMSSD/SCL rising this minute?)

Reframes the question from "predict the value" to "predict the
direction of change" — removes individual baseline differences,
targets the part of the signal that actually moves with behavior.

In [ ]:
sort_cols = ["PP", "Condition"] + (["timestamp"] if "timestamp" in df.columns else [])
df_asp = df.sort_values(sort_cols).copy()

for col in PHY_COLS:
    df_asp[col + "_delta"] = df_asp.groupby(["PP", "Condition"])[col].diff()
    df_asp[col + "_rising"] = (df_asp[col + "_delta"] > 0).astype(float)
for col in HCI_COLS:
    df_asp[col + "_delta"] = df_asp.groupby(["PP", "Condition"])[col].diff()
df_asp = df_asp.dropna(subset=[c + "_rising" for c in PHY_COLS])

HCI_DELTA = [c + "_delta" for c in HCI_COLS]
groups_asp = df_asp["PP"].to_numpy()

print(f"Rows with valid delta targets: {len(df_asp)}\n")

results_direction = {}
for tgt in PHY_COLS:
    y = df_asp[tgt + "_rising"].to_numpy(dtype=int)
    X = np.nan_to_num(df_asp[HCI_DELTA].to_numpy(dtype=float))
    accs = []
    for held in sorted(set(groups_asp)):
        tr = groups_asp != held; te = groups_asp == held
        if tr.sum() < 20 or te.sum() < 5:
            continue
        Xtr, ytr = X[tr], y[tr]; Xte, yte = X[te], y[te]
        mu = Xtr.mean(0); sd = Xtr.std(0) + 1e-9
        m = rf(); m.fit((Xtr - mu) / sd, ytr)
        accs.append(accuracy_score(yte, m.predict((Xte - mu) / sd)))
    results_direction[tgt] = np.mean(accs)
    print(f"{tgt:8s} direction accuracy = {results_direction[tgt]:.3f}  (chance = 0.50)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(results_direction.keys(), results_direction.values(),
       color=["steelblue", "tomato", "#2ca02c"])
ax.axhline(0.5, color="k", ls="--", label="chance")
ax.set_ylabel("LOSO accuracy"); ax.set_title("Direction: rising or falling?")
ax.legend()
plt.tight_layout(); plt.show()

## Experiment D — Magnitude class (strong fall / flat / strong rise)

Richer than binary direction — also captures how much the signal
changed, using each person's own delta variability as the threshold.

In [ ]:
results_magnitude = {}
for tgt in PHY_COLS:
    delta_col = tgt + "_delta"
    df_asp["_sd"] = df_asp.groupby("PP")[delta_col].transform("std")
    df_asp["_mag"] = 1
    df_asp.loc[df_asp[delta_col] > 0.5 * df_asp["_sd"], "_mag"] = 2
    df_asp.loc[df_asp[delta_col] < -0.5 * df_asp["_sd"], "_mag"] = 0
    y = df_asp["_mag"].to_numpy(dtype=int)
    X = np.nan_to_num(df_asp[HCI_DELTA].to_numpy(dtype=float))
    accs = []
    for held in sorted(set(groups_asp)):
        tr = groups_asp != held; te = groups_asp == held
        if tr.sum() < 20 or te.sum() < 5:
            continue
        Xtr, ytr = X[tr], y[tr]; Xte, yte = X[te], y[te]
        mu = Xtr.mean(0); sd = Xtr.std(0) + 1e-9
        m = rf(); m.fit((Xtr - mu) / sd, ytr)
        accs.append(accuracy_score(yte, m.predict((Xte - mu) / sd)))
    results_magnitude[tgt] = np.mean(accs)
    print(f"{tgt:8s} magnitude accuracy = {results_magnitude[tgt]:.3f}  (chance = 0.333)")

## Experiment E — CCA: shared latent cognitive-load dimension

Instead of asking "does one HCI feature predict one physiology
feature," CCA finds whether a *combination* of HCI features
correlates with a *combination* of physiology features. This is the
strongest result in the whole analysis.

In [ ]:
hci_z = [c + "_z" for c in HCI_COLS]
phy_z = [c + "_z" for c in PHY_COLS]
agg_clean = agg[hci_z + phy_z + ["Condition"]].dropna()
print(f"Clean rows for CCA: {len(agg_clean)}")

X_cca = agg_clean[hci_z].to_numpy(float)
Y_cca = agg_clean[phy_z].to_numpy(float)
cond_lbl = agg_clean["Condition"].to_numpy()

n_comp = min(3, Y_cca.shape[1])
cca = CCA(n_components=n_comp, max_iter=2000)
X_c, Y_c = cca.fit_transform(X_cca, Y_cca)
canons = [np.corrcoef(X_c[:, i], Y_c[:, i])[0, 1] for i in range(n_comp)]

print("\nCanonical correlations (training data):")
for i, r in enumerate(canons):
    print(f"  Component {i+1}: r = {r:.3f}")

cmap = {"N": "#4dac26", "I": "#d01c8b", "T": "#f1a340"}
fig, axes = plt.subplots(1, n_comp, figsize=(5 * n_comp, 4))
if n_comp == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    for cond in COND_ORDER:
        m = cond_lbl == cond
        ax.scatter(X_c[m, i], Y_c[m, i], label=COND_NAMES[cond], s=60, alpha=0.85)
    ax.set_xlabel(f"HCI canonical variate {i+1}")
    ax.set_ylabel(f"Physio canonical variate {i+1}")
    ax.set_title(f"Component {i+1}  (r={canons[i]:.2f})")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Cross-validate — checks whether the strong correlation above is real
# or an artifact of having relatively few rows (75) versus features (18).
cv = ShuffleSplit(n_splits=50, test_size=0.2, random_state=42)
cv_r = [[] for _ in range(n_comp)]
for tr_idx, te_idx in cv.split(X_cca):
    cca_cv = CCA(n_components=n_comp, max_iter=2000)
    cca_cv.fit(X_cca[tr_idx], Y_cca[tr_idx])
    Xte_c, Yte_c = cca_cv.transform(X_cca[te_idx], Y_cca[te_idx])
    for i in range(n_comp):
        cv_r[i].append(np.corrcoef(Xte_c[:, i], Yte_c[:, i])[0, 1])

print("Cross-validated canonical correlations (50 ShuffleSplit folds):")
for i in range(n_comp):
    print(f"  Component {i+1}: CV r = {np.nanmean(cv_r[i]):.3f} +/- "
          f"{np.nanstd(cv_r[i]):.3f}  (train r = {canons[i]:.3f})")

In [ ]:
# Which HCI features actually drive the shared dimension?
hci_weights = cca.x_weights_
phy_weights = cca.y_weights_
print("Top HCI contributors per component:\n")
for i in range(n_comp):
    hw = hci_weights[:, i]
    order = np.argsort(np.abs(hw))[::-1][:5]
    parts = [f"{HCI_COLS[k]}({hw[k]:+.2f})" for k in order]
    pw = phy_weights[:, i]
    p_order = np.argsort(np.abs(pw))[::-1]
    p_parts = [f"{PHY_COLS[k]}({pw[k]:+.2f})" for k in p_order]
    print(f"Component {i+1}:")
    print(f"  physiology: {', '.join(p_parts)}")
    print(f"  HCI:        {', '.join(parts)}\n")

## Summary

This notebook is intentionally self-contained — every result above
comes purely from SWELL-KW's own native columns. No external dataset
or proxy model was used to fill in or augment anything here, and none
is needed: SWELL-KW's own behavior-physiology relationship already
stands on its own (see the CCA cross-validation above for the honest,
non-overfit version of that claim).

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Experiment B  - HCI condition classifier accuracy: {hci_accs.mean():.3f}  (chance 0.333)")
print(f"Experiment C  - best direction accuracy: {max(results_direction.values()):.3f}  (chance 0.50)")
print(f"Experiment D  - best magnitude accuracy: {max(results_magnitude.values()):.3f}  (chance 0.333)")
print(f"Experiment E  - CCA component 1: train r={canons[0]:.3f}, CV r={np.nanmean(cv_r[0]):.3f}")